# Persona Engine — Stage 5: Context + Behavioral Representation Pipeline

**Objective:** Build and audit the Stage 5 dataset engineering pipeline for **Context + Behavioral Representation**.

**Core Directives:**
- Frozen Stage 4 v1 baseline is preserved intact.
- Raw data and Stage 4 splits remain 100% immutable.
- Model fine-tuning is **intentionally NOT performed yet**.

---
### Pipeline Steps
1. Hardware verification & package dependencies
2. Execute complete unit test suite across all feature extractors (`behavioral_features`, `topic_classifier`, `context_features`)
3. Build Stage 5 annotated datasets & ambiguous prompts benchmark
4. Execute strict Stage 5 dataset audit (0 missing targets, 0 modifications, 0 leakage, 0 duplicates)
5. Render audit report & examine behavioral distributions
6. Inspect ambiguous prompts benchmark

## 1. Hardware & Environment Diagnostics

In [ ]:
!nvidia-smi
import sys, torch
print('Python Version  :', sys.version)
print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name :', torch.cuda.get_device_name(0))
    print('Total VRAM (GB) :', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

## 2. Environment Setup & Dependency Installation

In [ ]:
# Install dependencies for Stage 5 pipeline
!pip install -q pyyaml jsonschema datasets

## 3. Repository Directory Verification

In [ ]:
import os
from pathlib import Path

# If persona-engine is cloned into /content/persona-engine, change directory
if Path('/content/persona-engine').is_dir():
    os.chdir('/content/persona-engine')

print('Current Working Directory:', Path.cwd())
assert (Path.cwd() / 'ml/scripts/build_stage5_dataset.py').is_file(), 'Error: ml/scripts/build_stage5_dataset.py not found!'
assert (Path.cwd() / 'ml/scripts/audit_stage5_dataset.py').is_file(), 'Error: ml/scripts/audit_stage5_dataset.py not found!'

## 4. Run Complete Unit Test Suite

In [ ]:
# Run all 50 unit tests across feature extraction, topic classifier, context dynamics, and pipeline
!python -m unittest discover -s ml/tests

## 5. Build Stage 5 Datasets & Benchmarks

In [ ]:
# Execute Stage 5 dataset engineering pipeline
!python ml/scripts/build_stage5_dataset.py

## 6. Execute Stage 5 Dataset Audit

In [ ]:
# Run strict integrity audit and distribution profiling
!python ml/scripts/audit_stage5_dataset.py

## 7. Render Audit Report

In [ ]:
from IPython.display import Markdown, display
with open('ml/data/stage5/reports/stage5_dataset_audit.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

## 8. Inspect Ambiguous Prompts Benchmark

In [ ]:
import json
with open('ml/data/stage5/benchmarks/ambiguous_prompts.jsonl', 'r', encoding='utf-8') as f:
    benchmarks = [json.loads(line) for line in f]

print(f"Total Ambiguous Prompts Identified: {len(benchmarks)}\n")
for b in benchmarks[:5]:
    print(f"Prompt: '{b['prompt']}' | {b['variant_count']} distinct targets across {b['total_occurrences']} occurrences")
    print(f"  Target Variants: {b['unique_target_responses'][:4]}")
    print("-" * 60)